## Test X.ai API (GROQ)

In [ ]:
import os
from openai import OpenAI
import openai

openai.api_key = os.getenv("XAI_API_KEY")

client = OpenAI(
  api_key=openai.api_key ,
  base_url="https://api.x.ai/v1",
)

completion = client.chat.completions.create(
  model="grok-3-latest",
  messages=[
    {"role": "system", "content": "You are a PhD-level mathematician."},
    {"role": "user", "content": "What is 2 + 2?"},
  ],
)

print(completion.choices[0].message)

In [ ]:
from git import Repo
import tempfile
import os

# URL of the repo you want to ingest
REPO_URL = "https://github.com/RWTH-EBC/AixLib.git"

# Clone into a temp folder
tmp_dir = tempfile.mkdtemp()
Repo.clone_from(REPO_URL, tmp_dir)
print(f"Cloned into {tmp_dir}")

In [ ]:
import glob
import os

# 読み込み対象の拡張子
EXTENSIONS = [".md", ".py", ".txt", ".rst"]

def load_repo_texts(root_dir):
    docs = []
    for ext in EXTENSIONS:
        pattern = os.path.join(root_dir, "**", f"*{ext}")
        for path in glob.glob(pattern, recursive=True):
            # ファイルでなければスキップ
            if not os.path.isfile(path):
                continue
            # サイズが大きすぎるものはスキップ
            if os.path.getsize(path) > 1e6:
                continue
            try:
                with open(path, encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                docs.append({
                    "path": os.path.relpath(path, root_dir),
                    "content": text
                })
            except Exception as e:
                # 万が一の読み込みエラーも無視
                print(f"Warning: failed to read {path}: {e}")
    return docs

# 使い方
documents = load_repo_texts(tmp_dir)
print(f"Loaded {len(documents)} files")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = []
for doc in documents:
    texts = splitter.split_text(doc["content"])
    for i, txt in enumerate(texts):
        chunks.append({
            "id": f"{doc['path']}-{i}",
            "text": txt
        })

print(f"Created {len(chunks)} text chunks")

In [ ]:
import openai
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

openai.api_key = os.getenv("XAI_API_KEY")
embedder = OpenAIEmbeddings()

# texts: list of strings
texts = [c["text"] for c in chunks]
metadatas = [{"source": c["id"]} for c in chunks]

# Create FAISS index
index = FAISS.from_texts(texts, embedder, metadatas=metadatas)

In [ ]:
!ls

In [1]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
import openai

# ↓ すでに保存してある faiss_index フォルダを読み込む
embedder = OpenAIEmbeddings()
index = FAISS.load_local(
    "../faiss_index",
    embedder,
    allow_dangerous_deserialization=True  # <-- これを追加
)

/var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/ipykernel_33843/4183473670.py:6: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedder = OpenAIEmbeddings()


In [ ]:
index

In [4]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
import openai
import os

openai.api_key = os.getenv("XAI_API_KEY")
# LLM
llm = ChatOpenAI(model_name="grok-3-beta",   
                 api_key=openai.api_key ,
                 base_url="https://api.x.ai/v1",
                 temperature=0)

# RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # or "map_reduce", "refine", etc.
    retriever=index.as_retriever(),
    return_source_documents=True
)

# Ask a question
query = "Where is heat exchanger models in this project?"
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

Answer:
 The heat exchanger models in this project are located under the following paths:

- **AixLib.Fluid.HeatExchangers**: This directory contains various heat exchanger models, including specific implementations like SensibleCooler_T, WetCoilEffectivenessNTU, ActiveBeams (with subcategories for Cooling, CoolingAndHeating, and related base classes), and others.
- **AixLib.Fluid.HeatExchangers.Examples**: This includes example models and base classes for heat exchangers, such as DryCoilEffectivenessNTUPControl, WaterCooler_T, WaterHeater_T, and WetCoilEffectivenessNTUMassFlow.
- **AixLib.Fluid.HeatExchangers.Radiators**: Contains radiator models like RadiatorEN442_2 and related examples.
- **AixLib.Fluid.HeatExchangers.Validation**: Includes validation models for heat exchangers, such as ConstantEffectiveness, DryCoilEffectivenessNTU, EvaporatorCondenser, and WetCoilEffectivenessNTU.
- Additionally, there are references to evaporator and condenser models in the **PythonModel.heatexch

In [5]:
# Ask a question
query = "Construct room air conditioner model by AixLib."
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

Answer:
 To construct a room air conditioner model using **AixLib**, you can leverage the library's components for HVAC systems, particularly focusing on models related to compressors, heat exchangers, and air handling. AixLib, developed at RWTH Aachen University's E.ON Energy Research Center, provides a comprehensive set of models for building performance simulations, including HVAC systems. Below, I will outline a general approach to constructing a room air conditioner model using components from AixLib. Since the provided context does not include a specific pre-built air conditioner model, we will build one using relevant sub-components.

### Step-by-Step Guide to Construct a Room Air Conditioner Model in AixLib

1. **Understand the Components of a Room Air Conditioner**:
   A typical room air conditioner (split or window unit) operates on a vapor-compression refrigeration cycle and includes:
   - A compressor (to compress the refrigerant).
   - An evaporator (to cool the room air b

## LangGraph agent

In [ ]:
!uv add langgraph

In [ ]:
from langchain.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, START
from typing import TypedDict

# 1) State 定義
class AgentState(TypedDict):
    instruction: str
    components: list[dict]
    plan: str
    model_code: str

# 2) LLM クライアント
llm = ChatOpenAI(
    model_name="grok-3-beta",
    api_key=openai.api_key,
    base_url="https://api.x.ai/v1",
    temperature=0
)

# 3) RetrievalQA チェーンは既存コードを流用

# ノード関数
async def retrieve_components(state: AgentState) -> AgentState:
    query = f"List the component definitions relevant to: {state['instruction']}"
    result = await qa.arun(query)
    state['components'] = parse_components(result)
    return state

async def plan_model_step(state: AgentState) -> AgentState:
    prompt = (
        "Given these components:\n"
        f"{state['components']}\n"
        f"And the user wants: {state['instruction']}\n"
        "Generate a step-by-step plan to assemble the Modelica model."
    )
    state['plan'] = await llm.apredict(messages=[{"role":"user","content":prompt}])
    return state

async def generate_model_code(state: AgentState) -> AgentState:
    prompt = (
        "Using the plan below and the components provided, produce the full Modelica model code:\n"
        f"Plan:\n{state['plan']}\nComponents:\n{state['components']}\n"
    )
    state['model_code'] = await llm.apredict(messages=[{"role":"user","content":prompt}])
    return state

# 4) グラフ構築（ノード名を一意に）
graph = StateGraph(AgentState)
graph.add_node("retrieve_components", retrieve_components)
graph.add_node("plan_model_step",     plan_model_step)
graph.add_node("generate_model_code", generate_model_code)

# エッジ設定
graph.add_edge(START,               "retrieve_components")
graph.add_edge("retrieve_components","plan_model_step")
graph.add_edge("plan_model_step",    "generate_model_code")

In [ ]:
# 5) 実行
compiled = graph.compile()
initial: AgentState = {
    "instruction":  "Construct room air conditioner model.",
    "components":   [],
    "plan":         "",
    "model_code":   ""
}
result = compiled.invoke(initial)
print(result["model_code"])

In [ ]:
# 省略: graph.compile() まで同じ準備

compiled = graph.compile()

initial: AgentState = {
    "instruction":  "Construct room air conditioner model.",
    "components":   [],
    "plan":         "",
    "model_code":   ""
}

# ← ここだけ変更
result = await compiled.ainvoke(initial)
print(result["model_code"])